In [6]:
%env LAB_OPEN_ROUTER_API_KEY=[REDACTED_OPENROUTER_KEY]


env: LAB_OPEN_ROUTER_API_KEY=[REDACTED_OPENROUTER_KEY]


In [9]:
from openai import OpenAI
from openai import OpenAIError
from dotenv import load_dotenv
import os
import time

load_dotenv()

API_KEY=os.getenv("LAB_OPEN_ROUTER_API_KEY")
assert API_KEY is not None, "LAB_OPEN_ROUTER_API_KEY is not set"

# safely expose it to the OpenAI SDK
os.environ["OPENAI_API_KEY"] = API_KEY

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=API_KEY,
)


def patient_risk_simulation_oss(model, prompt, max_retries = 3):
    for attempt in range (1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0,
                messages=[
                    {
                        "role": "user",
                        "content": (
                            "Assign a numerical risk score from 0–10 (10 being the highest risk) " 
                            "for hospitalization for a person with the following characteristics: "
                            f"{prompt}. "
                            "Respond only with:\n"
                            "1. Risk Assessment Score\n"
                            "2. A list of every single parameter and their associated numerical positive or negative risk values "
                            "that add up to the Risk Assessment Score\n"
                            "3. A one to two sentence rationale.\n"
                            "Exclude baseline risk in all calculations."
                        )
                    }
                ]
            )

            # catching errors
            text = response.choices[0].message.content
            return text if text is not None else ""
            
        except OpenAIError as e:
            # Don't retry on rate limits
            if "429" in str(e) or "Rate limit" in str(e):
                raise
            time.sleep(2 * attempt)

    
    # if all retries failed...    
    return ""
    #return response.choices[0].message.content


SyntaxError: invalid syntax (3687624538.py, line 47)

In [3]:
import os
import pandas as pd

df = pd.read_csv("../synthetic_data1.csv")

# change these
num_patients = 1 # originally 50
sim_reps = 3 # originally 2
folder_name = "full_sim_test" #"full_sim_run_1"

os.makedirs(folder_name, exist_ok=True)

results = []

for patient_id in range(num_patients):
    row = df.iloc[patient_id]
    prompt = ", ".join([f"{col}: {row[col]}" for col in df.columns])

    for sim_num in range(sim_reps):
        out = {
            "Patient_ID": patient_id,
            "Simulation_Number": sim_num
        }

        # ---- Model call ----
        result = patient_risk_simulation_oss(
            "openai/gpt-oss-20b:free",
            prompt=prompt
        )

        # ---- Write raw output (no overwrite) ---- <- this is causing silently empty .txt results -SB
        # file_path = os.path.join(
        #     folder_name, f"output_{patient_id}_{sim_num}.txt"
        # )
        # with open(file_path, "w", encoding="utf-8") as f:
        #     f.write(result)

        ## this way, you will get a file saying "[EMPTY RESPONSE]" (which we can debug) if the model does something that would result in an empty file:
        file_path = os.path.join(
            folder_name, f"output_{patient_id}_{sim_num}.txt"
        )
        try:
            result = patient_risk_simulation_oss(
                "openai/gpt-oss-20b:free",
                prompt=patient_prompt
            )
        except Exception as e:
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(f"[API_ERROR]\n{type(e).__name__}: {e}\n")
            print(f"API error for patient {patient_idx}, sim {sim_idx}")
            continue

        if result and result.strip():
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(result)
        else:
            with open(file_path, "w", encoding="utf-8") as f:
                f.write("[EMPTY_RESPONSE]\n")
        print(f"Wrote output for patient {patient_id}, sim {sim_num}") #small sanity check


        # ---- Parse response ----
        text = result.replace("–", "-").strip()
        lines = [line.strip() for line in text.splitlines() if line.strip()]

        # ---- 1. Risk Assessment Score ----
        out["Risk_Assessment_Score"] = None
        for idx, line in enumerate(lines):
            if "Risk Assessment Score" in line:
                for look_ahead in range(idx + 1, min(idx + 4, len(lines))):
                    candidate = lines[look_ahead].replace("*", "").strip()
                    try:
                        out["Risk_Assessment_Score"] = float(candidate)
                        break
                    except ValueError:
                        continue
                break

        # ---- 2. Parameter table ----
        table_start = None
        for idx, line in enumerate(lines):
            if "Parameter" in line and "Value" in line:
                table_start = idx + 1
                break

        if table_start is not None:
            for line in lines[table_start:]:
                if not line.startswith("|"):
                    break

                parts = [p.strip() for p in line.split("|") if p.strip()]
                if len(parts) == 2:
                    name, val = parts
                    try:
                        out[name] = float(val)
                    except ValueError:
                        pass

        # ---- 3. Rationale ----
        rationale_lines = []
        in_rationale = False
        for line in lines:
            if "Rationale" in line:
                in_rationale = True
                continue
            if in_rationale:
                rationale_lines.append(line)

        out["Rationale"] = " ".join(rationale_lines).strip()

        results.append(out)
        print(out)

# ---- Write final CSV ----
pd.DataFrame(results).to_csv(
    "parsed_output_fill_in_nones.csv",
    index=False
)


Wrote output for patient 0, sim 0
{'Patient_ID': 0, 'Simulation_Number': 0, 'Risk_Assessment_Score': 6.0, 'Rationale': 'The individual’s schizophrenia and severe depression markedly elevate hospitalization risk, while smoking, alcohol use, and a family history of heart disease add further medical burden. Protective factors such as regular exercise, healthy diet, social engagement, and civic activities offset some risk, yielding an overall moderate risk score of 6 on a 0-10 scale.'}
Wrote output for patient 1, sim 0
{'Patient_ID': 1, 'Simulation_Number': 0, 'Risk_Assessment_Score': 6.0, 'Rationale': 'The individual’s severe depression, smoking habit, low socioeconomic status, lack of exercise, and limited transportation options elevate hospitalization risk, while other lifestyle factors remain neutral, yielding a moderate overall risk score of 6.'}


In [5]:
import os
import pandas as pd


# Load data and prepare output
# -------------------------
df = pd.read_csv("../synthetic_data1.csv")
num_patients = 50
sim_reps = 5
folder_name = "full_sim_run_1"
os.makedirs(folder_name, exist_ok=True)

results = []

# -------------------------
# Main loop
# -------------------------
for patient_idx in range(num_patients):
    row = df.iloc[patient_idx]
    patient_prompt = ", ".join([f"{col}: {row[col]}" for col in df.columns])

    for sim_idx in range(sim_reps):
        out = {
            "Patient_ID": patient_idx,
            "Simulation_Number": sim_idx
        }

        # ---- GPT-OSS call ----
        result = patient_risk_simulation_oss(
            model="openai/gpt-oss-20b:free",  # GPT-OSS model
            prompt=patient_prompt,
            #client=client
        )

        # Save raw output
        file_path = os.path.join(
            folder_name, f"output_patient{patient_idx}_sim{sim_idx}.txt"
        )
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(result)

        # ---- Parsing logic (unchanged) ----
        text = result.replace("–", "-").strip()
        lines = [line.strip() for line in text.splitlines() if line.strip()]

        # 1. Risk score
        score_val = None
        for i, line in enumerate(lines):
            if "Risk Assessment Score" in line:
                for j in range(i + 1, min(i + 4, len(lines))):
                    try:
                        score_val = float(lines[j])
                        break
                    except ValueError:
                        continue
                break
        out["Risk_Assessment_Score"] = score_val

        # 2. Parameter table
        start = None
        for i, line in enumerate(lines):
            if "Parameter" in line and "Value" in line:
                start = i + 1
                break

        if start is not None:
            for line in lines[start:]:
                if not line.startswith("|"):
                    break
                parts = [p.strip() for p in line.split("|") if p.strip()]
                if len(parts) == 2:
                    name, val = parts
                    try:
                        out[name] = float(val)
                    except ValueError:
                        pass

        # 3. Rationale
        rationale = []
        capture = False
        for line in lines:
            if "Rationale" in line:
                capture = True
                continue
            if capture:
                rationale.append(line)
        out["Rationale"] = " ".join(rationale)

        results.append(out)
        print(f"Completed patient {patient_idx}, sim {sim_idx}")

# -------------------------
# Save all results
# -------------------------
pd.DataFrame(results).to_csv("parsed_output_fill_in_nones.csv", index=False)


Completed patient 0, sim 0
Completed patient 0, sim 1
Completed patient 0, sim 2
Completed patient 0, sim 3
Completed patient 0, sim 4
Completed patient 1, sim 0
Completed patient 1, sim 1
Completed patient 1, sim 2
Completed patient 1, sim 3
Completed patient 1, sim 4
Completed patient 2, sim 0
Completed patient 2, sim 1
Completed patient 2, sim 2
Completed patient 2, sim 3
Completed patient 2, sim 4
Completed patient 3, sim 0
Completed patient 3, sim 1
Completed patient 3, sim 2
Completed patient 3, sim 3
Completed patient 3, sim 4
Completed patient 4, sim 0
Completed patient 4, sim 1
Completed patient 4, sim 2
Completed patient 4, sim 3
Completed patient 4, sim 4
Completed patient 5, sim 0
Completed patient 5, sim 1
Completed patient 5, sim 2
Completed patient 5, sim 3
Completed patient 5, sim 4
Completed patient 6, sim 0
Completed patient 6, sim 1
Completed patient 6, sim 2
Completed patient 6, sim 3
Completed patient 6, sim 4
Completed patient 7, sim 0
Completed patient 7, sim 1
C

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1768867200000'}, 'provider_name': None}}, 'user_id': 'user_38SiswbdqQGGhrWs4HjqTxzSBhS'}